# Notebook 01 — Ingeniería de datos

**Objetivo de este notebook:** transformar el dump SQL raw en datos limpios y estructurados
que los notebooks de análisis puedan consumir directamente.

La ingeniería de datos es el trabajo que nadie ve pero que hace posible todo lo demás.
Un análisis construido sobre datos sucios va a dar resultados incorrectos, sin importar
qué tan sofisticado sea el modelo.

**Al terminar este notebook vas a tener:**
- 5 archivos `.parquet` limpios en `results/ironmarch/`: posts, usuarios, hilos, foros, PMs
- Un corpus por usuario listo para análisis semántico (embeddings y NER)
- Una tabla usuario↔thread lista para construir el grafo de red

**Formato Parquet:** es el formato estándar para guardar DataFrames grandes en ciencia de datos.
Es más eficiente que CSV: ocupa menos espacio, carga más rápido, y preserva los tipos de datos
(fechas, enteros, strings) sin conversiones adicionales.

## 1. Imports y configuración

Las mismas librerías que en el notebook anterior más `matplotlib` y `seaborn` para
visualizar el resultado de la limpieza.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_forum, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)
IM_ZIP = DATA_DIR / 'Far Right Forum' / 'IronMarch_2019.11.zip'

print(f'ZIP existe: {IM_ZIP.exists()}')
print(f'Resultados: {IM_RESULTS}')

## 2. Carga raw

Usamos `load_forum()` para cargar todas las tablas desde el ZIP.
Esta función hace el trabajo pesado de parsear el SQL de IPS 4.x y retornar
DataFrames de pandas — no necesitamos saber SQL para usarla.

**¿Por qué volvemos a cargar si ya lo hicimos en el notebook 00?**
Cada notebook es independiente — puede ejecutarse por separado sin depender de otros.
Esto es una convención importante en Jupyter: el estado no se transfiere entre notebooks.

In [ ]:
# Cargar todas las tablas
tables = load_forum(IM_ZIP)

posts_raw   = tables.get('post', pd.DataFrame())
users_raw   = tables.get('user', pd.DataFrame())
threads_raw = tables.get('thread', pd.DataFrame())
forums      = tables.get('forum', pd.DataFrame())
pms_raw     = tables.get('private_message', pd.DataFrame())

print('=== Tamaño de las tablas raw (antes de limpiar) ===')
print(f'  posts:    {posts_raw.shape[0]:>7,} filas  ×  {posts_raw.shape[1]} columnas')
print(f'  users:    {users_raw.shape[0]:>7,} filas  ×  {users_raw.shape[1]} columnas')
print(f'  threads:  {threads_raw.shape[0]:>7,} filas  ×  {threads_raw.shape[1]} columnas')
print(f'  forums:   {forums.shape[0]:>7,} filas  ×  {forums.shape[1]} columnas')
print(f'  pms:      {pms_raw.shape[0]:>7,} filas  ×  {pms_raw.shape[1]} columnas')

print('\n=== Tipos de columnas — posts ===')
print(posts_raw.dtypes)

print('\n=== Rango de fechas — posts raw ===')
if 'dateline' in posts_raw.columns:
    # Las fechas vienen como timestamps Unix (segundos desde 1970-01-01)
    dates = pd.to_datetime(posts_raw['dateline'], errors='coerce', utc=True)
    print(f'  Mínima: {dates.min()}')
    print(f'  Máxima: {dates.max()}')
    print(f'  Fechas en 1970 (epoch=0): {(dates.dt.year == 1970).sum():,}')

`load_forum()` (de `src/utils.py`) auto-detecta el motor del foro por el contenido del ZIP y no requiere saber de antemano si es vBulletin, MyBB o IPS.

Ver [`src/README.md`](../src/README.md) para el resto de la API de `src/utils.py`.

## 3. Pipeline de limpieza

Aplicamos la misma secuencia de pasos a cada tabla. Los pasos de limpieza son:

1. **Posts:** eliminar texto nulo o vacío, deduplicar por `postid`, descartar posts muy cortos
   (menos de 10 caracteres), filtrar fechas imposibles (año 1970 = timestamp Unix cero).
2. **Usuarios:** deduplicar por `userid`, descartar sin `username`, normalizar nombre a minúsculas.
3. **Hilos:** deduplicar por `threadid`.
4. **Foros:** sin cambios (tabla de referencia pequeña).
5. **Mensajes privados:** eliminar sin texto o sin fecha.

Después de cada paso imprimimos cuántas filas se eliminaron para tener trazabilidad.
Si un paso elimina más del 20% de filas, es señal de que hay un problema de calidad en el dato original.

In [ ]:
# === LIMPIEZA DE POSTS ===
n_raw_posts = len(posts_raw)
posts_clean = posts_raw.copy()

# Paso 1: detectar qué columna tiene el texto (IPS usa 'pagetext')
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'

# Paso 2: eliminar posts sin texto
posts_clean = posts_clean[posts_clean[text_col].notna()]
posts_clean = posts_clean[posts_clean[text_col].str.strip() != '']
print(f'Posts sin texto eliminados: {n_raw_posts - len(posts_clean):,}')

# Paso 3: deduplicar por postid (clave primaria de posts)
before = len(posts_clean)
posts_clean = posts_clean.drop_duplicates(subset='postid')
print(f'Duplicados de postid eliminados: {before - len(posts_clean):,}')

# Paso 4: normalizar userid a numérico
# El parser SQL a veces genera filas desalineadas donde el userid contiene texto
posts_clean['userid'] = pd.to_numeric(posts_clean['userid'], errors='coerce')
before = len(posts_clean)
posts_clean = posts_clean.dropna(subset=['userid'])
posts_clean['userid'] = posts_clean['userid'].astype(int).astype(str)
print(f'Posts con userid inválido eliminados: {before - len(posts_clean):,}')

# Paso 5: convertir fechas y filtrar epoch-0 (1970) y años < 2000
posts_clean['dateline'] = pd.to_datetime(posts_clean['dateline'], utc=True, errors='coerce')
before = len(posts_clean)
posts_clean = posts_clean[posts_clean['dateline'].notna()]
posts_clean = posts_clean[posts_clean['dateline'].dt.year >= 2000]
print(f'Posts con fecha inválida/epoch-0 eliminados: {before - len(posts_clean):,}')

# Paso 6: filtrar posts muy cortos (menos de 10 caracteres)
before = len(posts_clean)
posts_clean = posts_clean[posts_clean[text_col].str.len() >= 10]
print(f'Posts con menos de 10 caracteres eliminados: {before - len(posts_clean):,}')

posts_clean = posts_clean.reset_index(drop=True)
n_clean_posts = len(posts_clean)
print(f'\nPosts raw: {n_raw_posts:,}  →  Posts limpios: {n_clean_posts:,}')
print(f'Retención: {n_clean_posts / n_raw_posts * 100:.1f}%')

In [ ]:
# === LIMPIEZA DE USUARIOS ===
n_raw_users = len(users_raw)
users_clean = users_raw.copy()

# Normalizar userid a numérico
users_clean['userid'] = pd.to_numeric(users_clean['userid'], errors='coerce')
users_clean = users_clean.dropna(subset=['userid'])
users_clean['userid'] = users_clean['userid'].astype(int).astype(str)

# Deduplicar por userid
users_clean = users_clean.drop_duplicates(subset='userid')

# Eliminar usuarios sin nombre
users_clean = users_clean[users_clean['username'].notna()]

# Conservar el nombre original y agregar versión normalizada
# 'username_raw' tiene el nombre con mayúsculas/minúsculas originales
# 'username' es la versión en minúsculas para comparar sin sensibilidad a caso
users_clean = users_clean.rename(columns={'username': 'username_raw'})
users_clean['username'] = users_clean['username_raw'].str.strip().str.lower()

# Convertir fecha de registro
if 'joindate' in users_clean.columns:
    users_clean['joindate'] = pd.to_datetime(users_clean['joindate'], utc=True, errors='coerce')

users_clean = users_clean.reset_index(drop=True)
n_clean_users = len(users_clean)
print(f'Usuarios raw: {n_raw_users:,}  →  Usuarios limpios: {n_clean_users:,}')
print(f'Retención: {n_clean_users / n_raw_users * 100:.1f}%')

In [ ]:
# === LIMPIEZA DE HILOS Y MENSAJES PRIVADOS ===
n_raw_threads = len(threads_raw)
threads_clean = threads_raw.copy()

# Normalizar threadid
threads_clean['threadid'] = pd.to_numeric(threads_clean['threadid'], errors='coerce')
threads_clean = threads_clean.dropna(subset=['threadid'])
threads_clean['threadid'] = threads_clean['threadid'].astype(int).astype(str)

threads_clean = threads_clean.drop_duplicates(subset='threadid').reset_index(drop=True)
n_clean_threads = len(threads_clean)
print(f'Threads raw: {n_raw_threads:,}  →  Threads limpios: {n_clean_threads:,}')

# Mensajes privados
n_raw_pms = len(pms_raw)
pms_clean = pms_raw.copy()

# Detectar columna de texto en PMs
pm_text_col = 'text' if 'text' in pms_clean.columns else (
    'message' if 'message' in pms_clean.columns else None
)
if pm_text_col:
    pms_clean = pms_clean[pms_clean[pm_text_col].notna()]
    pms_clean = pms_clean[pms_clean[pm_text_col].str.strip() != '']

pms_clean['dateline'] = pd.to_datetime(
    pms_clean.get('dateline', pd.Series(dtype='object')), utc=True, errors='coerce'
) if 'dateline' in pms_clean.columns else pms_clean.get('dateline')
if 'dateline' in pms_clean.columns:
    pms_clean = pms_clean[pms_clean['dateline'].notna()]

pms_clean = pms_clean.reset_index(drop=True)
n_clean_pms = len(pms_clean)
print(f'PMs raw: {n_raw_pms:,}  →  PMs limpios: {n_clean_pms:,}')

In [ ]:
# Verificar que el destinatario (receiver_id) llegó bien desde el join
# core_message_posts.msg_topic_id ↔ core_message_topics.mt_id (ver src/parsers/ips.py)
print('Columnas en pms_clean:', pms_clean.columns.tolist())
if 'receiver_id' in pms_clean.columns:
    null_receiver = pms_clean['receiver_id'].isna().sum()
    print(f'PMs sin receiver_id: {null_receiver:,} de {len(pms_clean):,}')
    print('\nEjemplo (remitente → destinatario):')
    print(pms_clean[['sender_id', 'receiver_id', 'dateline']].head())

## 4. Estadísticas post-limpieza

Antes de guardar, visualizamos el efecto de la limpieza para verificar que no eliminamos
demasiado dato válido. La gráfica superpone posts por día antes y después — deben verse
muy similares salvo por el relleno de fechas imposibles al principio.

In [ ]:
# Tabla resumen de limpieza
summary = pd.DataFrame([
    {'Tabla': 'posts',   'Filas raw': n_raw_posts,   'Filas limpias': n_clean_posts,
     'Eliminadas': n_raw_posts - n_clean_posts,
     'Retención %': f'{n_clean_posts/n_raw_posts*100:.1f}%'},
    {'Tabla': 'users',   'Filas raw': n_raw_users,   'Filas limpias': n_clean_users,
     'Eliminadas': n_raw_users - n_clean_users,
     'Retención %': f'{n_clean_users/n_raw_users*100:.1f}%'},
    {'Tabla': 'threads', 'Filas raw': n_raw_threads, 'Filas limpias': n_clean_threads,
     'Eliminadas': n_raw_threads - n_clean_threads,
     'Retención %': f'{n_clean_threads/n_raw_threads*100:.1f}%'},
    {'Tabla': 'forums',  'Filas raw': len(forums),   'Filas limpias': len(forums),
     'Eliminadas': 0,    'Retención %': '100.0%'},
    {'Tabla': 'pms',     'Filas raw': n_raw_pms,     'Filas limpias': n_clean_pms,
     'Eliminadas': n_raw_pms - n_clean_pms,
     'Retención %': f'{n_clean_pms/n_raw_pms*100:.1f}%'},
])
print(summary.to_string(index=False))

# Gráfica: posts por día antes vs después de limpiar
fig, ax = plt.subplots(figsize=(14, 4))

raw_dates = pd.to_datetime(posts_raw['dateline'], utc=True, errors='coerce').dropna()
raw_daily = raw_dates.dt.floor('D').value_counts().sort_index()
clean_daily = posts_clean['dateline'].dt.floor('D').value_counts().sort_index()

ax.fill_between(raw_daily.index, raw_daily.values,   alpha=0.35, label='raw',   color='#8888ff')
ax.fill_between(clean_daily.index, clean_daily.values, alpha=0.65, label='limpio', color='#44aaff')
ax.set_title('Posts por día — antes y después de la limpieza')
ax.set_xlabel('Fecha')
ax.set_ylabel('Posts')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Preparación para análisis estructural (grafo de red)

Para construir el grafo de co-participación (notebook 02), necesitamos una tabla simple
que diga: "este usuario participó en este hilo". Esto se construye cruzando posts con hilos.

La estructura que generamos es:
```
userid | threadid | forumid
```

Cada fila representa que un usuario participó en un hilo. Las aristas del grafo se
construirán conectando usuarios que compartieron al menos un hilo.

In [ ]:
# Construir tabla usuario↔thread para el grafo
# Solo necesitamos userid y threadid de posts
user_thread = posts_clean[['userid', 'threadid']].copy()
user_thread['threadid'] = user_thread['threadid'].astype(str)

# Agregar forumid usando el join con hilos
if 'forumid' in threads_clean.columns:
    thread_to_forum = threads_clean.set_index('threadid')['forumid'].to_dict()
    user_thread['forumid'] = user_thread['threadid'].map(thread_to_forum)

# Deduplicar: si un usuario escribió 10 posts en el mismo hilo,
# para la red solo cuenta como una participación
user_thread_dedup = user_thread.drop_duplicates(subset=['userid', 'threadid'])

print(f'Participaciones únicas usuario↔thread: {len(user_thread_dedup):,}')
print(f'Usuarios involucrados: {user_thread_dedup["userid"].nunique():,}')
print(f'Hilos involucrados:    {user_thread_dedup["threadid"].nunique():,}')
print(f'\nEjemplo de las primeras filas:')
print(user_thread_dedup.head())

# Estadística: ¿cuántos hilos por usuario?
threads_per_user = user_thread_dedup.groupby('userid').size()
print(f'\nHistograma de hilos por usuario:')
print(f'  Mediana: {threads_per_user.median():.0f} hilos/usuario')
print(f'  Máximo:  {threads_per_user.max():,} hilos (usuario más activo en red)')

## 6. Preparación para análisis semántico (corpus por usuario)

Para los embeddings y NER (notebook 03), necesitamos un **corpus por usuario**:
la concatenación de todos los posts de cada usuario en un solo texto.

También generamos una variante con solo los posts más largos porque:
- Los posts cortos suelen ser respuestas triviales ("de acuerdo", "+1", emojis)
- Los posts largos son donde el usuario expresa sus ideas con más detalle
- Los modelos de embeddings tienen límites de contexto — no podemos pasar texto ilimitado

Guardamos los **top-100 posts más largos por usuario** como balance entre
cobertura y costo computacional (ver análisis en notebook 03).

In [ ]:
import re

def clean_html(text: str) -> str:
    """Eliminar etiquetas HTML y URLs del texto de un post."""
    text = re.sub(r'<[^>]+>', ' ', str(text))  # Quitar tags HTML
    text = re.sub(r'https?://\S+', ' ', text)   # Quitar URLs
    text = re.sub(r'\s+', ' ', text)             # Normalizar espacios
    return text.strip()

# Aplicar limpieza de HTML al texto
posts_clean['text_clean'] = posts_clean[text_col].astype(str).apply(clean_html)

# Filtrar posts que quedaron vacíos después de quitar HTML
posts_clean = posts_clean[posts_clean['text_clean'].str.len() > 20]

# Calcular longitud de cada post (útil para el muestreo)
posts_clean['text_len'] = posts_clean['text_clean'].str.len()

# Top-100 posts más largos por usuario
top100_per_user = (
    posts_clean
    .sort_values('text_len', ascending=False)  # Ordenar de mayor a menor
    .groupby('userid')
    .head(100)  # Tomar los primeros 100 de cada usuario
    .reset_index(drop=True)
)

print(f'Posts para análisis semántico (top-100 por usuario):')
print(f'  Posts totales seleccionados: {len(top100_per_user):,}')
print(f'  Usuarios cubiertos: {top100_per_user["userid"].nunique():,}')
print(f'  Longitud media del post seleccionado: {top100_per_user["text_len"].mean():.0f} chars')
print(f'  Longitud máxima: {top100_per_user["text_len"].max():,} chars')

# Distribución de posts por usuario en la muestra
posts_per_user = top100_per_user.groupby('userid').size()
print(f'\n  Usuarios con exactamente 100 posts en muestra: '
      f'{(posts_per_user == 100).sum():,} (los más prolíficos)')
print(f'  Usuarios con menos de 100 posts (todos sus posts): '
      f'{(posts_per_user < 100).sum():,}')

In [ ]:
# Corpus concatenado por usuario (todos los posts, limitado a 50K chars)
# Este límite evita que un usuario con millones de palabras bloquee el modelo de embeddings
MAX_CHARS = 50_000

corpus_by_user = (
    posts_clean
    .groupby('userid')['text_clean']
    .apply(lambda texts: ' '.join(texts.tolist())[:MAX_CHARS])
    .reset_index()
    .rename(columns={'text_clean': 'corpus'})
)

# Agregar nombre de usuario
uid_to_name = dict(zip(users_clean['userid'], users_clean['username_raw']))
corpus_by_user['username'] = corpus_by_user['userid'].map(uid_to_name)
corpus_by_user['corpus_len'] = corpus_by_user['corpus'].str.len()

# Filtrar usuarios con muy poco texto (insuficiente para análisis semántico)
MIN_CORPUS = 500  # 500 caracteres = ~100 palabras mínimo
corpus_by_user = corpus_by_user[corpus_by_user['corpus_len'] >= MIN_CORPUS]

print(f'Corpus por usuario generado:')
print(f'  Usuarios con corpus ≥ {MIN_CORPUS} chars: {len(corpus_by_user):,}')
print(f'  Usuarios que alcanzan el límite de 50K chars: {(corpus_by_user["corpus_len"] >= MAX_CHARS).sum():,}')
print(f'  Longitud media del corpus: {corpus_by_user["corpus_len"].mean():,.0f} chars')

print(f'\nTop 5 usuarios por tamaño de corpus:')
print(corpus_by_user.nlargest(5, 'corpus_len')[['username', 'corpus_len']].to_string(index=False))

## 7. Guardar parquets limpios

Guardamos todas las tablas limpias en formato **Parquet** (`.parquet`).
Los notebooks 02 y 03 van a cargar desde estos archivos en lugar de re-parsear
el ZIP cada vez — esto ahorra 1-2 minutos de carga.

`df.to_parquet(path, index=False)` guarda el DataFrame en disco.
- `path` es la ruta donde guardar el archivo
- `index=False` le dice que no guarde el índice numérico de pandas (no lo necesitamos)

In [ ]:
# Guardar todas las tablas limpias
tablas = [
    ('posts',        posts_clean),
    ('users',        users_clean),
    ('threads',      threads_clean),
    ('forums',       forums),
    ('pms',          pms_clean),
    ('user_thread',  user_thread_dedup),
    ('corpus_users', corpus_by_user),
    ('top100_posts', top100_per_user),
]

for name, df in tablas:
    path = IM_RESULTS / f'{name}_clean.parquet'
    df.to_parquet(path, index=False)
    size_kb = path.stat().st_size / 1024
    print(f'  {name}_clean.parquet: {len(df):,} filas  |  {size_kb:.0f} KB')

print(f'\nTodos los archivos guardados en: {IM_RESULTS}')

## 8. Validación de los parquets guardados

Recargamos los archivos que acabamos de guardar y verificamos que:
1. El número de filas coincide con lo que guardamos
2. Las columnas clave no tienen valores nulos
3. El rango de fechas es correcto (ninguna fecha en 1970)

Esta verificación es obligatoria en pipelines de datos. Un archivo corrupto o truncado
puede pasar desapercibido sin esta comprobación.

In [ ]:
print('=== VALIDACIÓN DE PARQUETS ===\n')
checks_ok = True

# Verificar conteos
for name, df_original in tablas:
    path = IM_RESULTS / f'{name}_clean.parquet'
    df_loaded = pd.read_parquet(path)
    match = len(df_loaded) == len(df_original)
    status = 'OK' if match else 'ERROR'
    print(f'[{name}]  original={len(df_original):,}  recargado={len(df_loaded):,}  [{status}]')
    if not match:
        checks_ok = False

# Verificar columnas críticas en posts
print()
posts_v = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')

null_postid = posts_v['postid'].isna().sum() if 'postid' in posts_v.columns else 'N/A'
null_userid = posts_v['userid'].isna().sum() if 'userid' in posts_v.columns else 'N/A'
min_year    = posts_v['dateline'].dt.year.min() if 'dateline' in posts_v.columns else 0

print(f'[posts] null postid:  {null_postid}  [{"OK" if null_postid == 0 else "ERROR"}]')
print(f'[posts] null userid:  {null_userid}  [{"OK" if null_userid == 0 else "ERROR"}]')
print(f'[posts] año mínimo:   {min_year}  [{"OK" if min_year >= 2000 else "ERROR — hay fechas 1970"}]')

# Verificar usuarios
users_v = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')
null_uname = users_v['username'].isna().sum() if 'username' in users_v.columns else 'N/A'
print(f'[users] null username: {null_uname}  [{"OK" if null_uname == 0 else "ERROR"}]')

print()
if checks_ok:
    print('=== VALIDACIÓN PASADA — todos los parquets son correctos ===')
else:
    print('=== VALIDACIÓN FALLIDA — revisar los errores arriba ===')

print(f'\nSiguiente paso: 02_analisis_estructural.ipynb')